# 33｜手写 TextCNN 与 Hierarchical Attention Network 文档分类

同一个文档分类任务可以有两种完全不同的归纳偏置：TextCNN 用多尺度卷积抓局部 n-gram；HAN 先在句内编码词，再在篇章内编码句子，并输出两级可审计注意力。本笔记不调用 `nn.RNN/LSTM/GRU`，而是手写 GRU cell、双向扫描、masked attention、`forward`、训练与可信制品。

> 合成数据只证明实现可以学习一个已知规则，不代表真实文档分类效果或注意力具有因果解释性。

## 1. 两套输入合同

- TextCNN：`tokens/mask: [B,T]`，有效 token 为左对齐前缀；长度必须不小于最大卷积核。
- HAN：`tokens/word_mask: [B,S,W]`；有效句子左对齐，每个有效句内部的词也左对齐。padding 句允许全空，但每篇文档至少一个有效句。
- TextCNN 输出 `[B,C]`；HAN 额外返回 `word_alpha: [B,S,W]` 和 `sent_alpha: [B,S]`。
- 有效 attention 权重和为 1，padding 句/词权重为 0；全 padding 文档必须 fail closed。
- 固定 CPU 和随机种子，不联网，不调用任何预制循环层。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import copy  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。

import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 20260812  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。
PAD = 0  # 计算并保存当前步骤的中间状态。
VOCAB_SIZE = 40  # 计算并保存当前步骤的中间状态。
NUM_CLASSES = 3  # 计算并保存当前步骤的中间状态。
CLASSIFIER_TOKENS = ["<pad>"] + [f"token_{index}" for index in range(1, VOCAB_SIZE)]  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert VOCAB_SIZE > 10 and NUM_CLASSES == 3  # 用受控断言验证关键不变量。
assert len(CLASSIFIER_TOKENS) == VOCAB_SIZE and len(set(CLASSIFIER_TOKENS)) == VOCAB_SIZE  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "seed": SEED})  # 执行当前语句以推进本节示例。

## 2. 统一验证变长与空 padding

长度错误若拖到卷积、softmax 或 gather 才暴露，报错通常很难定位。我们先验证 prefix mask：`True,True,False` 合法，`True,False,True` 非法。HAN 的 padding 句可以全 False，但有效句之后不能再次出现有效句。全空文档没有可归一化的句级分布，因此立即拒绝。

In [ ]:
def validate_prefix_rows(mask, allow_empty=False, name="mask"):  # 定义本节可复用的核心函数。
    if mask.dtype != torch.bool or mask.ndim != 2:  # 按当前条件选择后续控制路径。
        raise ValueError(f"{name} 必须是二维 bool")  # 遇到非法合同立即显式失败。
    if not allow_empty and bool((~mask.any(1)).any()):  # 按当前条件选择后续控制路径。
        raise ValueError(f"{name} 每行至少一个有效位置")  # 遇到非法合同立即显式失败。
    if mask.shape[1] > 1 and bool((mask[:, 1:] & ~mask[:, :-1]).any()):  # 按当前条件选择后续控制路径。
        raise ValueError(f"{name} 必须是左对齐前缀")  # 遇到非法合同立即显式失败。


def masked_softmax(scores, mask):  # 定义本节可复用的核心函数。
    if scores.shape != mask.shape or mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
        raise ValueError("masked_softmax 形状或类型错误")  # 遇到非法合同立即显式失败。
    masked_scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)  # 计算并保存当前步骤的中间状态。
    row_has_value = mask.any(-1, keepdim=True)  # 计算并保存当前步骤的中间状态。
    safe_max = torch.where(row_has_value, masked_scores.max(-1, keepdim=True).values, torch.zeros_like(scores[:, :1]))  # 计算并保存当前步骤的中间状态。
    numer = torch.exp(masked_scores - safe_max) * mask  # 计算并保存当前步骤的中间状态。
    weights = numer / numer.sum(-1, keepdim=True).clamp_min(torch.finfo(scores.dtype).tiny)  # 计算并保存当前步骤的中间状态。
    return weights  # 返回当前分支计算出的结果。

scores0 = torch.tensor([[1.0, 2.0, 99.0], [3.0, 4.0, 5.0]])  # 计算并保存当前步骤的中间状态。
mask_test = torch.tensor([[True, True, False], [False, False, False]])  # 计算并保存当前步骤的中间状态。
w0 = masked_softmax(scores0, mask_test)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(w0[0].sum(), torch.tensor(1.0))  # 用受控断言验证关键不变量。
assert torch.count_nonzero(w0[0, 2:]) == 0 and torch.count_nonzero(w0[1]) == 0  # 用受控断言验证关键不变量。

## 3. TextCNN：多尺度 n-gram 与 max-over-time

对窗口宽度 $k$：

$$c_i^{(k)}=\operatorname{ReLU}(W_k\,x_{i:i+k-1}+b_k),\qquad
\hat c^{(k)}=\max_{i\in\text{valid windows}}c_i^{(k)}.$$

多个 `k` 的 pooled feature 拼接后分类。关键不是 `Conv1d` 本身，而是**只允许全部由有效 token 构成的窗口参与最大值**；否则 padding embedding 或卷积 bias 会成为长度特征。

In [ ]:
class TextCNN(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size, embed_dim, channels, kernel_sizes, num_classes):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if not kernel_sizes or min(kernel_sizes) <= 0 or len(set(kernel_sizes)) != len(kernel_sizes):  # 按当前条件选择后续控制路径。
            raise ValueError("kernel_sizes 必须是非空正整数集合")  # 遇到非法合同立即显式失败。
        self.config = dict(vocab_size=vocab_size, embed_dim=embed_dim, channels=channels,  # 计算并保存当前步骤的中间状态。
                           kernel_sizes=list(kernel_sizes), num_classes=num_classes)  # 计算并保存当前步骤的中间状态。
        self.max_kernel = max(kernel_sizes)  # 计算并保存当前步骤的中间状态。
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)  # 计算并保存当前步骤的中间状态。
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, channels, k) for k in kernel_sizes])  # 计算并保存当前步骤的中间状态。
        self.classifier = nn.Linear(channels * len(kernel_sizes), num_classes)  # 计算并保存当前步骤的中间状态。

    def forward(self, tokens, mask):  # 定义本节可复用的核心函数。
        if tokens.ndim != 2 or tokens.dtype != torch.long or mask.shape != tokens.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("TextCNN tokens/mask 合同错误")  # 遇到非法合同立即显式失败。
        validate_prefix_rows(mask, allow_empty=False, name="TextCNN mask")  # 计算并保存当前步骤的中间状态。
        lengths = mask.sum(1)  # 计算并保存当前步骤的中间状态。
        if bool((lengths < self.max_kernel).any()):  # 按当前条件选择后续控制路径。
            raise ValueError("有效长度不能小于最大卷积核")  # 遇到非法合同立即显式失败。
        if tokens.numel() and (int(tokens.min()) < 0 or int(tokens.max()) >= self.config["vocab_size"]):  # 按当前条件选择后续控制路径。
            raise ValueError("token id 越界")  # 遇到非法合同立即显式失败。
        embedded = self.embedding(tokens).transpose(1, 2)  # 计算并保存当前步骤的中间状态。
        pooled = []  # 计算并保存当前步骤的中间状态。
        for conv, kernel in zip(self.convs, self.config["kernel_sizes"]):  # 遍历输入元素以累积或检查结果。
            features = F.relu(conv(embedded))  # 计算并保存当前步骤的中间状态。
            valid_windows = mask.unfold(1, kernel, 1).all(-1)  # 计算并保存当前步骤的中间状态。
            features = features.masked_fill(~valid_windows.unsqueeze(1), torch.finfo(features.dtype).min)  # 计算并保存当前步骤的中间状态。
            pooled.append(features.max(-1).values)  # 执行当前语句以推进本节示例。
        document_features = torch.cat(pooled, dim=-1)  # 计算并保存当前步骤的中间状态。
        return self.classifier(document_features), document_features  # 返回当前分支计算出的结果。

textcnn_probe = TextCNN(VOCAB_SIZE, 12, 8, [2, 3, 4], NUM_CLASSES)  # 计算并保存当前步骤的中间状态。
tc_tokens = torch.tensor([[4, 5, 6, 7, PAD, PAD], [8, 9, 10, 11, 12, PAD]])  # 计算并保存当前步骤的中间状态。
tc_mask = tc_tokens.ne(PAD)  # 计算并保存当前步骤的中间状态。
tc_logits, tc_features = textcnn_probe(tc_tokens, tc_mask)  # 计算并保存当前步骤的中间状态。
assert tc_logits.shape == (2, NUM_CLASSES) and tc_features.shape == (2, 24)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    textcnn_probe(torch.tensor([[4, 5, PAD, PAD]]), torch.tensor([[True, True, False, False]]))  # 执行当前语句以推进本节示例。
    raise AssertionError("过短序列应被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "最大卷积核" in str(exc)  # 用受控断言验证关键不变量。

## 4. 手写 GRU cell 与双向变长扫描

使用更新门、重置门和候选状态：

$$z_t=\sigma(W_zx_t+U_zh_{t-1}),\quad r_t=\sigma(W_rx_t+U_rh_{t-1}),$$
$$\tilde h_t=\tanh(W_nx_t+r_t\odot U_nh_{t-1}),\quad
h_t=z_t\odot h_{t-1}+(1-z_t)\odot\tilde h_t.$$

双向扫描各用一套 cell。padding 步不更新状态且输出清零；反向扫描仍按原 mask 判断每个位置，而不是把 padding 当作序列开头。

In [ ]:
class ScratchGRUCell(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_size, hidden_size):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if input_size <= 0 or hidden_size <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("GRU 维度必须为正")  # 遇到非法合同立即显式失败。
        self.input_size, self.hidden_size = input_size, hidden_size  # 计算并保存当前步骤的中间状态。
        self.x_proj = nn.Linear(input_size, 3 * hidden_size, bias=True)  # 计算并保存当前步骤的中间状态。
        self.h_proj = nn.Linear(hidden_size, 3 * hidden_size, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, x_t, h_prev):  # 定义本节可复用的核心函数。
        if x_t.ndim != 2 or h_prev.ndim != 2 or x_t.shape[0] != h_prev.shape[0]:  # 按当前条件选择后续控制路径。
            raise ValueError("GRU 单步输入形状错误")  # 遇到非法合同立即显式失败。
        if x_t.shape[1] != self.input_size or h_prev.shape[1] != self.hidden_size:  # 按当前条件选择后续控制路径。
            raise ValueError("GRU 特征维错误")  # 遇到非法合同立即显式失败。
        x_z, x_r, x_n = self.x_proj(x_t).chunk(3, -1)  # 计算并保存当前步骤的中间状态。
        h_z, h_r, h_n = self.h_proj(h_prev).chunk(3, -1)  # 计算并保存当前步骤的中间状态。
        z = torch.sigmoid(x_z + h_z)  # 计算并保存当前步骤的中间状态。
        r = torch.sigmoid(x_r + h_r)  # 计算并保存当前步骤的中间状态。
        candidate = torch.tanh(x_n + r * h_n)  # 计算并保存当前步骤的中间状态。
        return z * h_prev + (1.0 - z) * candidate  # 返回当前分支计算出的结果。


class ScratchBiGRU(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_size, hidden_size):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.hidden_size = hidden_size  # 计算并保存当前步骤的中间状态。
        self.forward_cell = ScratchGRUCell(input_size, hidden_size)  # 计算并保存当前步骤的中间状态。
        self.backward_cell = ScratchGRUCell(input_size, hidden_size)  # 计算并保存当前步骤的中间状态。

    def _scan(self, x, mask, cell, reverse=False):  # 定义本节可复用的核心函数。
        B, T, _ = x.shape  # 计算并保存当前步骤的中间状态。
        h = x.new_zeros(B, self.hidden_size)  # 计算并保存当前步骤的中间状态。
        outputs = [None] * T  # 计算并保存当前步骤的中间状态。
        indices = range(T - 1, -1, -1) if reverse else range(T)  # 计算并保存当前步骤的中间状态。
        for t in indices:  # 遍历输入元素以累积或检查结果。
            proposed = cell(x[:, t], h)  # 计算并保存当前步骤的中间状态。
            active = mask[:, t:t + 1]  # 计算并保存当前步骤的中间状态。
            h = torch.where(active, proposed, h)  # 计算并保存当前步骤的中间状态。
            outputs[t] = torch.where(active, h, torch.zeros_like(h))  # 计算并保存当前步骤的中间状态。
        return torch.stack(outputs, dim=1)  # 返回当前分支计算出的结果。

    def forward(self, x, mask):  # 定义本节可复用的核心函数。
        if x.ndim != 3 or mask.shape != x.shape[:2]:  # 按当前条件选择后续控制路径。
            raise ValueError("BiGRU 输入形状错误")  # 遇到非法合同立即显式失败。
        validate_prefix_rows(mask, allow_empty=True, name="BiGRU mask")  # 计算并保存当前步骤的中间状态。
        forward = self._scan(x, mask, self.forward_cell, reverse=False)  # 计算并保存当前步骤的中间状态。
        backward = self._scan(x, mask, self.backward_cell, reverse=True)  # 计算并保存当前步骤的中间状态。
        return torch.cat([forward, backward], dim=-1)  # 返回当前分支计算出的结果。

bigru_probe = ScratchBiGRU(5, 4)  # 计算并保存当前步骤的中间状态。
x_probe = torch.randn(2, 4, 5)  # 计算并保存当前步骤的中间状态。
m_probe = torch.tensor([[True, True, False, False], [False, False, False, False]])  # 计算并保存当前步骤的中间状态。
h_probe = bigru_probe(x_probe, m_probe)  # 计算并保存当前步骤的中间状态。
assert h_probe.shape == (2, 4, 8)  # 用受控断言验证关键不变量。
assert torch.count_nonzero(h_probe[0, 2:]) == 0 and torch.count_nonzero(h_probe[1]) == 0  # 用受控断言验证关键不变量。

## 5. HAN：词级与句级两次 attention

加性注意力先计算 $u_i=\tanh(Wh_i+b)$、$e_i=v^\top u_i$，再仅对有效位置 softmax：

$$\alpha_i=\frac{\exp(e_i)}{\sum_{j\in valid}\exp(e_j)},\qquad s=\sum_i\alpha_i h_i.$$

第一层把词表示聚合成句向量，第二层把句向量聚合成文档向量。padding 句的词权重总和为 0；有效句为 1；每篇有效文档的句权重为 1。注意力权重是模型内部路由，不自动等于因果解释。

In [ ]:
class AdditiveAttention(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_dim, attention_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.proj = nn.Linear(input_dim, attention_dim)  # 计算并保存当前步骤的中间状态。
        self.context = nn.Linear(attention_dim, 1, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, states, mask):  # 定义本节可复用的核心函数。
        if states.ndim != 3 or mask.shape != states.shape[:2]:  # 按当前条件选择后续控制路径。
            raise ValueError("attention 输入形状错误")  # 遇到非法合同立即显式失败。
        scores = self.context(torch.tanh(self.proj(states))).squeeze(-1)  # 计算并保存当前步骤的中间状态。
        weights = masked_softmax(scores, mask)  # 计算并保存当前步骤的中间状态。
        pooled = torch.sum(states * weights.unsqueeze(-1), dim=1)  # 计算并保存当前步骤的中间状态。
        return pooled, weights  # 返回当前分支计算出的结果。


class HierarchicalAttentionNetwork(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size, embed_dim, word_hidden, sentence_hidden, attention_dim, num_classes):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.config = dict(vocab_size=vocab_size, embed_dim=embed_dim, word_hidden=word_hidden,  # 计算并保存当前步骤的中间状态。
                           sentence_hidden=sentence_hidden, attention_dim=attention_dim,  # 计算并保存当前步骤的中间状态。
                           num_classes=num_classes)  # 计算并保存当前步骤的中间状态。
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)  # 计算并保存当前步骤的中间状态。
        self.word_encoder = ScratchBiGRU(embed_dim, word_hidden)  # 计算并保存当前步骤的中间状态。
        self.word_attention = AdditiveAttention(2 * word_hidden, attention_dim)  # 计算并保存当前步骤的中间状态。
        self.sentence_encoder = ScratchBiGRU(2 * word_hidden, sentence_hidden)  # 计算并保存当前步骤的中间状态。
        self.sentence_attention = AdditiveAttention(2 * sentence_hidden, attention_dim)  # 计算并保存当前步骤的中间状态。
        self.classifier = nn.Linear(2 * sentence_hidden, num_classes)  # 计算并保存当前步骤的中间状态。

    def forward(self, tokens, word_mask):  # 定义本节可复用的核心函数。
        if tokens.ndim != 3 or tokens.dtype != torch.long or word_mask.shape != tokens.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("HAN tokens/word_mask 合同错误")  # 遇到非法合同立即显式失败。
        if tokens.numel() and (int(tokens.min()) < 0 or int(tokens.max()) >= self.config["vocab_size"]):  # 按当前条件选择后续控制路径。
            raise ValueError("token id 越界")  # 遇到非法合同立即显式失败。
        B, S, W = tokens.shape  # 计算并保存当前步骤的中间状态。
        flat_mask = word_mask.reshape(B * S, W)  # 计算并保存当前步骤的中间状态。
        validate_prefix_rows(flat_mask, allow_empty=True, name="word mask")  # 计算并保存当前步骤的中间状态。
        sentence_mask = word_mask.any(-1)  # 计算并保存当前步骤的中间状态。
        validate_prefix_rows(sentence_mask, allow_empty=False, name="sentence mask")  # 计算并保存当前步骤的中间状态。

        embedded = self.embedding(tokens).reshape(B * S, W, -1)  # 计算并保存当前步骤的中间状态。
        word_states = self.word_encoder(embedded, flat_mask)  # 计算并保存当前步骤的中间状态。
        sentence_vectors, word_alpha = self.word_attention(word_states, flat_mask)  # 计算并保存当前步骤的中间状态。
        sentence_vectors = sentence_vectors.reshape(B, S, -1)  # 计算并保存当前步骤的中间状态。
        sentence_states = self.sentence_encoder(sentence_vectors, sentence_mask)  # 计算并保存当前步骤的中间状态。
        document_vector, sentence_alpha = self.sentence_attention(sentence_states, sentence_mask)  # 计算并保存当前步骤的中间状态。
        logits = self.classifier(document_vector)  # 计算并保存当前步骤的中间状态。
        return {"logits": logits, "document": document_vector,  # 返回当前分支计算出的结果。
                "word_alpha": word_alpha.reshape(B, S, W), "sentence_alpha": sentence_alpha,  # 执行当前语句以推进本节示例。
                "sentence_mask": sentence_mask}  # 执行当前语句以推进本节示例。

In [ ]:
han_probe = HierarchicalAttentionNetwork(VOCAB_SIZE, 10, 6, 7, 8, NUM_CLASSES)  # 计算并保存当前步骤的中间状态。
docs_probe = torch.tensor([  # 计算并保存当前步骤的中间状态。
    [[4, 5, 6, PAD], [7, 8, PAD, PAD], [PAD, PAD, PAD, PAD]],  # 执行当前语句以推进本节示例。
    [[9, 10, 11, 12], [PAD, PAD, PAD, PAD], [PAD, PAD, PAD, PAD]],  # 执行当前语句以推进本节示例。
])  # 执行当前语句以推进本节示例。
word_mask_probe = docs_probe.ne(PAD)  # 计算并保存当前步骤的中间状态。
han_out = han_probe(docs_probe, word_mask_probe)  # 计算并保存当前步骤的中间状态。
word_sums = han_out["word_alpha"].sum(-1)  # 计算并保存当前步骤的中间状态。
sent_sums = han_out["sentence_alpha"].sum(-1)  # 计算并保存当前步骤的中间状态。

assert han_out["logits"].shape == (2, NUM_CLASSES)  # 用受控断言验证关键不变量。
assert torch.allclose(word_sums[han_out["sentence_mask"]], torch.ones_like(word_sums[han_out["sentence_mask"]]))  # 用受控断言验证关键不变量。
assert torch.count_nonzero(word_sums[~han_out["sentence_mask"]]) == 0  # 用受控断言验证关键不变量。
assert torch.allclose(sent_sums, torch.ones_like(sent_sums))  # 用受控断言验证关键不变量。
assert torch.count_nonzero(han_out["sentence_alpha"][~han_out["sentence_mask"]]) == 0  # 用受控断言验证关键不变量。

all_pad = torch.zeros(1, 2, 3, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    han_probe(all_pad, all_pad.ne(PAD))  # 执行当前语句以推进本节示例。
    raise AssertionError("全 padding 文档应被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "sentence mask" in str(exc)  # 用受控断言验证关键不变量。

## 6. 同一批合成文档，两种模型公平对照

三类文档分别含有类标记 token `4/5/6`，其余为干扰词；句数与句长变化。我们从层级表示生成 TextCNN 的扁平前缀，并保持同一标签。数据切分在这里只用于受控过拟合；真实评估应按作者、主题、时间或文档来源切分，防止近重复文本泄漏。

In [ ]:
def make_documents(n_per_class=5, max_sentences=3, max_words=5):  # 定义本节可复用的核心函数。
    docs, labels = [], []  # 计算并保存当前步骤的中间状态。
    rng = random.Random(SEED + 20)  # 计算并保存当前步骤的中间状态。
    for label in range(NUM_CLASSES):  # 遍历输入元素以累积或检查结果。
        marker = 4 + label  # 计算并保存当前步骤的中间状态。
        for sample_index in range(n_per_class):  # 遍历输入元素以累积或检查结果。
            n_sentences = 1 + (sample_index % max_sentences)  # 计算并保存当前步骤的中间状态。
            doc = []  # 计算并保存当前步骤的中间状态。
            for s in range(n_sentences):  # 遍历输入元素以累积或检查结果。
                length = 4 + ((sample_index + s) % 2)  # 计算并保存当前步骤的中间状态。
                words = [7 + rng.randrange(VOCAB_SIZE - 7) for _ in range(length)]  # 计算并保存当前步骤的中间状态。
                if s == sample_index % n_sentences:  # 按当前条件选择后续控制路径。
                    words[(sample_index + s) % length] = marker  # 计算并保存当前步骤的中间状态。
                doc.append(words + [PAD] * (max_words - length))  # 执行当前语句以推进本节示例。
            doc += [[PAD] * max_words for _ in range(max_sentences - n_sentences)]  # 计算并保存当前步骤的中间状态。
            docs.append(doc); labels.append(label)  # 执行当前语句以推进本节示例。
    return torch.tensor(docs, dtype=torch.long), torch.tensor(labels, dtype=torch.long)  # 返回当前分支计算出的结果。


def flatten_documents(docs):  # 定义本节可复用的核心函数。
    rows = []  # 计算并保存当前步骤的中间状态。
    for doc in docs.tolist():  # 遍历输入元素以累积或检查结果。
        valid = [token for sentence in doc for token in sentence if token != PAD]  # 计算并保存当前步骤的中间状态。
        rows.append(valid)  # 执行当前语句以推进本节示例。
    max_len = max(len(row) for row in rows)  # 计算并保存当前步骤的中间状态。
    return torch.tensor([row + [PAD] * (max_len - len(row)) for row in rows], dtype=torch.long)  # 返回当前分支计算出的结果。

documents, document_labels = make_documents()  # 计算并保存当前步骤的中间状态。
document_word_mask = documents.ne(PAD)  # 计算并保存当前步骤的中间状态。
flat_documents = flatten_documents(documents)  # 计算并保存当前步骤的中间状态。
flat_mask = flat_documents.ne(PAD)  # 计算并保存当前步骤的中间状态。
assert documents.shape == (15, 3, 5) and document_labels.shape == (15,)  # 用受控断言验证关键不变量。
assert int(flat_mask.sum()) == int(document_word_mask.sum())  # 用受控断言验证关键不变量。
assert int(flat_mask.sum(1).min()) >= 4  # 用受控断言验证关键不变量。

## 7. 梯度合同与受控过拟合

训练前后在完全相同的 15 篇文档上比较交叉熵。我们分别检查 embedding、卷积/GRU、attention 和分类器能收到有限非零梯度。小集合训练准确率高只是一条“电路连通”证据；模型选择仍必须使用未参与优化的验证集。

In [ ]:
def train_to_overfit(model, batch_fn, labels, steps=120, lr=0.025):  # 定义本节可复用的核心函数。
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # 计算并保存当前步骤的中间状态。
    model.train()  # 执行当前语句以推进本节示例。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        initial_logits = batch_fn(model)  # 计算并保存当前步骤的中间状态。
        initial_loss = float(F.cross_entropy(initial_logits, labels))  # 计算并保存当前步骤的中间状态。
    first_gradients = None  # 计算并保存当前步骤的中间状态。
    for step in range(steps):  # 遍历输入元素以累积或检查结果。
        optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
        logits = batch_fn(model)  # 计算并保存当前步骤的中间状态。
        loss = F.cross_entropy(logits, labels)  # 计算并保存当前步骤的中间状态。
        loss.backward()  # 执行当前语句以推进本节示例。
        if step == 0:  # 按当前条件选择后续控制路径。
            first_gradients = {  # 计算并保存当前步骤的中间状态。
                name: float(parameter.grad.abs().sum())  # 执行当前语句以推进本节示例。
                for name, parameter in model.named_parameters()  # 遍历输入元素以累积或检查结果。
                if parameter.requires_grad and parameter.grad is not None  # 按当前条件选择后续控制路径。
            }  # 执行当前语句以推进本节示例。
            assert first_gradients and sum(first_gradients.values()) > 0  # 用受控断言验证关键不变量。
            assert all(torch.isfinite(parameter.grad).all() for parameter in model.parameters() if parameter.grad is not None)  # 用受控断言验证关键不变量。
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)  # 执行当前语句以推进本节示例。
        optimizer.step()  # 执行当前语句以推进本节示例。
    model.eval()  # 执行当前语句以推进本节示例。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        final_logits = batch_fn(model)  # 计算并保存当前步骤的中间状态。
        final_loss = float(F.cross_entropy(final_logits, labels))  # 计算并保存当前步骤的中间状态。
        accuracy = float((final_logits.argmax(-1) == labels).float().mean())  # 计算并保存当前步骤的中间状态。
    return initial_loss, final_loss, accuracy, first_gradients  # 返回当前分支计算出的结果。


torch.manual_seed(SEED + 1)  # 执行当前语句以推进本节示例。
textcnn = TextCNN(VOCAB_SIZE, embed_dim=12, channels=10, kernel_sizes=[2, 3, 4], num_classes=NUM_CLASSES)  # 计算并保存当前步骤的中间状态。
tc_metrics = train_to_overfit(  # 计算并保存当前步骤的中间状态。
    textcnn, lambda model: model(flat_documents, flat_mask)[0], document_labels, steps=70  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。

torch.manual_seed(SEED + 2)  # 执行当前语句以推进本节示例。
han = HierarchicalAttentionNetwork(VOCAB_SIZE, embed_dim=10, word_hidden=7,  # 计算并保存当前步骤的中间状态。
                                   sentence_hidden=7, attention_dim=8, num_classes=NUM_CLASSES)  # 计算并保存当前步骤的中间状态。
han_metrics = train_to_overfit(  # 计算并保存当前步骤的中间状态。
    han, lambda model: model(documents, document_word_mask)["logits"], document_labels, steps=90  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。

assert tc_metrics[1] < tc_metrics[0] * 0.15 and tc_metrics[2] == 1.0  # 用受控断言验证关键不变量。
assert han_metrics[1] < han_metrics[0] * 0.15 and han_metrics[2] == 1.0  # 用受控断言验证关键不变量。
for prefix in ["embedding", "convs", "classifier"]:  # 遍历输入元素以累积或检查结果。
    assert sum(value for name, value in tc_metrics[3].items() if name.startswith(prefix)) > 0, prefix  # 用受控断言验证关键不变量。
for prefix in ["embedding", "word_encoder", "word_attention", "sentence_encoder", "sentence_attention", "classifier"]:  # 遍历输入元素以累积或检查结果。
    assert sum(value for name, value in han_metrics[3].items() if name.startswith(prefix)) > 0, prefix  # 用受控断言验证关键不变量。
print({"TextCNN": [round(x, 4) for x in tc_metrics[:3]],  # 执行当前语句以推进本节示例。
       "HAN": [round(x, 4) for x in han_metrics[:3]]})  # 执行当前语句以推进本节示例。

## 8. 评估与解释边界

- 分类：accuracy、macro-F1、逐类 recall、混淆矩阵，并按文档长度、语言、来源切片。
- 稳健性：追加 padding、删除无关句、同义改写、标点/Unicode 归一化变化；这些不能与标签规则共享生成模板。
- 效率：TextCNN 各卷积约 $O(BT\,KEDC)$；HAN 的循环扫描约 $O(BSW(H^2+EH)+BSH_s^2)$，难像卷积一样完全并行。
- 注意力可用于调试“模型读了哪里”，但高权重不证明该词对预测具有因果贡献；需要遮挡、反事实或梯度方法交叉验证。

In [ ]:
textcnn.eval(); han.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    trained_han_out = han(documents, document_word_mask)  # 计算并保存当前步骤的中间状态。
    trained_word_sum = trained_han_out["word_alpha"].sum(-1)  # 计算并保存当前步骤的中间状态。
    trained_sent_sum = trained_han_out["sentence_alpha"].sum(-1)  # 计算并保存当前步骤的中间状态。
    valid_sentences = trained_han_out["sentence_mask"]  # 计算并保存当前步骤的中间状态。
assert torch.allclose(trained_word_sum[valid_sentences], torch.ones_like(trained_word_sum[valid_sentences]), atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.count_nonzero(trained_word_sum[~valid_sentences]) == 0  # 用受控断言验证关键不变量。
assert torch.allclose(trained_sent_sum, torch.ones_like(trained_sent_sum), atol=1e-6)  # 用受控断言验证关键不变量。

# 在固定 mask 下更改 padding token id，不得影响 HAN 的 logits。
pad_ids_changed = documents.clone()  # 计算并保存当前步骤的中间状态。
pad_ids_changed[~document_word_mask] = 17  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    original_logits = han(documents, document_word_mask)["logits"]  # 计算并保存当前步骤的中间状态。
    changed_logits = han(pad_ids_changed, document_word_mask)["logits"]  # 计算并保存当前步骤的中间状态。
assert torch.equal(original_logits, changed_logits)  # 用受控断言验证关键不变量。

bad_mask = torch.tensor([[True, False, True, False]])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    validate_prefix_rows(bad_mask, name="bad mask")  # 计算并保存当前步骤的中间状态。
    raise AssertionError("带洞 mask 应被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "左对齐" in str(exc)  # 用受控断言验证关键不变量。


# GRU 门数值 oracle：所有投影为 0 时 z=r=0.5、candidate=0，故 h_new=0.5*h_prev。
gate_oracle = ScratchGRUCell(3, 2).eval()  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    gate_oracle.x_proj.weight.zero_()  # 执行当前语句以推进本节示例。
    gate_oracle.x_proj.bias.zero_()  # 执行当前语句以推进本节示例。
    gate_oracle.h_proj.weight.zero_()  # 执行当前语句以推进本节示例。
gate_h = torch.tensor([[2.0, -4.0], [1.0, 3.0]])  # 计算并保存当前步骤的中间状态。
gate_out = gate_oracle(torch.randn(2, 3), gate_h)  # 计算并保存当前步骤的中间状态。
assert torch.equal(gate_out, 0.5 * gate_h)  # 用受控断言验证关键不变量。

# 反向扫描不能读取 padding embedding；TextCNN 也只能池化全有效窗口。
reverse_oracle = ScratchBiGRU(5, 4).eval()  # 计算并保存当前步骤的中间状态。
reverse_x = torch.randn(1, 5, 5)  # 计算并保存当前步骤的中间状态。
reverse_mask = torch.tensor([[True, True, True, False, False]])  # 计算并保存当前步骤的中间状态。
reverse_changed = reverse_x.clone()  # 计算并保存当前步骤的中间状态。
reverse_changed[:, 3:] = 999.0  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    reverse_a = reverse_oracle(reverse_x, reverse_mask)  # 计算并保存当前步骤的中间状态。
    reverse_b = reverse_oracle(reverse_changed, reverse_mask)  # 计算并保存当前步骤的中间状态。
assert torch.equal(reverse_a, reverse_b)  # 用受控断言验证关键不变量。

flat_pad_changed = flat_documents.clone()  # 计算并保存当前步骤的中间状态。
flat_pad_changed[~flat_mask] = VOCAB_SIZE - 1  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    textcnn_a = textcnn(flat_documents, flat_mask)[0]  # 计算并保存当前步骤的中间状态。
    textcnn_b = textcnn(flat_pad_changed, flat_mask)[0]  # 计算并保存当前步骤的中间状态。
assert torch.equal(textcnn_a, textcnn_b)  # 用受控断言验证关键不变量。

# 单独的 additive attention 对 state 置换保持 pooled 不变，weights 随位置同步置换。
permutation_attention = AdditiveAttention(6, 4).eval()  # 计算并保存当前步骤的中间状态。
permutation_states = torch.randn(2, 3, 6)  # 计算并保存当前步骤的中间状态。
permutation_mask = torch.ones(2, 3, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
permutation = torch.tensor([2, 0, 1])  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    pooled_a, alpha_a = permutation_attention(permutation_states, permutation_mask)  # 计算并保存当前步骤的中间状态。
    pooled_b, alpha_b = permutation_attention(  # 计算并保存当前步骤的中间状态。
        permutation_states[:, permutation], permutation_mask[:, permutation]  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
assert torch.allclose(pooled_a, pooled_b, atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(alpha_b, alpha_a[:, permutation], atol=1e-6)  # 用受控断言验证关键不变量。

# 完整 HAN 含句级 BiGRU，应对句序敏感；空句夹在有效句之间必须拒绝。
two_sentence_doc = documents[1:2]  # 计算并保存当前步骤的中间状态。
two_sentence_mask = document_word_mask[1:2]  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    order_a = han(two_sentence_doc, two_sentence_mask)["logits"]  # 计算并保存当前步骤的中间状态。
    order_b = han(two_sentence_doc[:, [1, 0, 2]], two_sentence_mask[:, [1, 0, 2]])["logits"]  # 计算并保存当前步骤的中间状态。
sentence_order_delta = float((order_a - order_b).abs().max())  # 计算并保存当前步骤的中间状态。
assert sentence_order_delta > 1e-6  # 用受控断言验证关键不变量。
middle_empty = documents[2:3].clone()  # 计算并保存当前步骤的中间状态。
middle_empty[:, 1] = PAD  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    han(middle_empty, middle_empty.ne(PAD))  # 执行当前语句以推进本节示例。
    raise AssertionError("有效句之间的空句必须被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "sentence mask" in str(exc)  # 用受控断言验证关键不变量。
print({"gru_gate_error": float((gate_out - 0.5 * gate_h).abs().max()),  # 执行当前语句以推进本节示例。
       "reverse_padding_error": float((reverse_a - reverse_b).abs().max()),  # 执行当前语句以推进本节示例。
       "textcnn_padding_error": float((textcnn_a - textcnn_b).abs().max()),  # 执行当前语句以推进本节示例。
       "han_sentence_order_delta": sentence_order_delta})  # 执行当前语句以推进本节示例。


## 9. 完整双模型制品与发布者 registry

TextCNN 与 HAN 必须分别绑定架构白名单、构造参数、**完整 token 顺序**、唯一且非空的 label map、各自输入表示与 padding 规则、实际训练文档/扁平输入/split、训练 recipe，以及权重的原始字节和 tensor 语义摘要。

package 内自带 SHA 不是信任根。下面由发布者侧只读 registry 保存 `(artifact_id, version) -> immutable manifest digest`；loader 先查 registry，再校验 state bytes，以及包含 tensor key/dtype/shape/bytes 的摘要。攻击者即使整体替换模型、词表或标签并重算内部 hash，也不能更新 registry。

In [ ]:
from types import MappingProxyType  # 导入本单元所需的依赖。

def json_hash(value):  # 定义本节可复用的核心函数。
    raw = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(raw).hexdigest()  # 返回当前分支计算出的结果。


def classifier_tensor_state_hash(state_dict):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state_dict):  # 遍历输入元素以累积或检查结果。
        tensor = state_dict[key]  # 计算并保存当前步骤的中间状态。
        if not isinstance(tensor, torch.Tensor):  # 按当前条件选择后续控制路径。
            raise TypeError("state_dict 只能包含 tensor")  # 遇到非法合同立即显式失败。
        value = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        descriptor = {"key": key, "dtype": str(value.dtype), "shape": list(value.shape)}  # 计算并保存当前步骤的中间状态。
        digest.update(json.dumps(descriptor, sort_keys=True, separators=(",", ":")).encode())  # 计算并保存当前步骤的中间状态。
        digest.update(value.numpy().tobytes(order="C"))  # 计算并保存当前步骤的中间状态。
    return digest.hexdigest()  # 返回当前分支计算出的结果。


def expected_preprocess(architecture, config):  # 定义本节可复用的核心函数。
    common = {"tokenizer": "synthetic-id-list-v1", "normalization": "identity-demo-v1",  # 计算并保存当前步骤的中间状态。
              "padding": "right", "pad_id": PAD}  # 执行当前语句以推进本节示例。
    if architecture == "textcnn":  # 按当前条件选择后续控制路径。
        return {**common, "representation": "flatten-row-major-valid-tokens",  # 返回当前分支计算出的结果。
                "kernel_sizes": list(config["kernel_sizes"]),  # 执行当前语句以推进本节示例。
                "min_valid_tokens": max(config["kernel_sizes"])}  # 执行当前语句以推进本节示例。
    if architecture == "han":  # 按当前条件选择后续控制路径。
        return {**common, "representation": "hierarchical-document",  # 返回当前分支计算出的结果。
                "max_sentences": 3, "max_words": 5,  # 执行当前语句以推进本节示例。
                "empty_sentence_policy": "trailing-only", "word_mask_true": "valid"}  # 执行当前语句以推进本节示例。
    raise ValueError("未知架构")  # 遇到非法合同立即显式失败。


def validate_classifier_manifest(manifest):  # 定义本节可复用的核心函数。
    required = {"schema", "artifact_id", "version", "subject", "architecture", "config",  # 计算并保存当前步骤的中间状态。
                "vocab", "label_map", "preprocess", "training_snapshot",  # 执行当前语句以推进本节示例。
                "state_bytes_sha256", "state_tensor_sha256"}  # 执行当前语句以推进本节示例。
    if not isinstance(manifest, dict) or set(manifest) != required:  # 按当前条件选择后续控制路径。
        raise ValueError("classifier manifest 字段不完整")  # 遇到非法合同立即显式失败。
    if manifest["schema"] != "document-classifier-v2" or not manifest["artifact_id"] or not manifest["version"]:  # 按当前条件选择后续控制路径。
        raise ValueError("artifact 身份字段错误")  # 遇到非法合同立即显式失败。
    architecture, config = manifest["architecture"], manifest["config"]  # 计算并保存当前步骤的中间状态。
    expected_config_keys = {  # 计算并保存当前步骤的中间状态。
        "textcnn": {"vocab_size", "embed_dim", "channels", "kernel_sizes", "num_classes"},  # 执行当前语句以推进本节示例。
        "han": {"vocab_size", "embed_dim", "word_hidden", "sentence_hidden",  # 执行当前语句以推进本节示例。
                "attention_dim", "num_classes"},  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    if architecture not in expected_config_keys or set(config) != expected_config_keys[architecture]:  # 按当前条件选择后续控制路径。
        raise ValueError("架构或 config 字段错误")  # 遇到非法合同立即显式失败。
    vocab = manifest["vocab"]  # 计算并保存当前步骤的中间状态。
    if set(vocab) != {"kind", "tokens", "pad_id"} or vocab["kind"] != "synthetic-id-list-v1":  # 按当前条件选择后续控制路径。
        raise ValueError("vocab schema 错误")  # 遇到非法合同立即显式失败。
    tokens = vocab["tokens"]  # 计算并保存当前步骤的中间状态。
    if not isinstance(tokens, list) or len(tokens) != len(set(tokens)):  # 按当前条件选择后续控制路径。
        raise ValueError("完整 token 顺序必须唯一")  # 遇到非法合同立即显式失败。
    if len(tokens) != config["vocab_size"] or vocab["pad_id"] != PAD or tokens[PAD] != "<pad>":  # 按当前条件选择后续控制路径。
        raise ValueError("vocab_size/pad/token 顺序不一致")  # 遇到非法合同立即显式失败。
    labels = manifest["label_map"]  # 计算并保存当前步骤的中间状态。
    if (not isinstance(labels, list) or len(labels) != config["num_classes"] or  # 按当前条件选择后续控制路径。
            len(labels) != len(set(labels)) or any(not isinstance(label, str) or not label for label in labels)):  # 计算并保存当前步骤的中间状态。
        raise ValueError("label 数量必须匹配 num_classes，且唯一非空")  # 遇到非法合同立即显式失败。
    if manifest["preprocess"] != expected_preprocess(architecture, config):  # 按当前条件选择后续控制路径。
        raise ValueError("预处理快照与架构不一致")  # 遇到非法合同立即显式失败。
    snapshot = manifest["training_snapshot"]  # 计算并保存当前步骤的中间状态。
    if snapshot.get("vocab_sha256") != json_hash(vocab):  # 按当前条件选择后续控制路径。
        raise ValueError("训练快照 vocab 指纹错误")  # 遇到非法合同立即显式失败。
    if snapshot.get("label_map_sha256") != json_hash(labels):  # 按当前条件选择后续控制路径。
        raise ValueError("训练快照 label map 指纹错误")  # 遇到非法合同立即显式失败。
    if snapshot.get("preprocess_sha256") != json_hash(manifest["preprocess"]):  # 按当前条件选择后续控制路径。
        raise ValueError("训练快照 preprocess 指纹错误")  # 遇到非法合同立即显式失败。
    dataset = snapshot.get("dataset", {})  # 计算并保存当前步骤的中间状态。
    hierarchical = dataset.get("hierarchical_input_ids")  # 计算并保存当前步骤的中间状态。
    flat = dataset.get("flat_input_ids")  # 计算并保存当前步骤的中间状态。
    target_labels = dataset.get("labels")  # 计算并保存当前步骤的中间状态。
    if not isinstance(hierarchical, list) or not hierarchical or not (  # 按当前条件选择后续控制路径。
        len(hierarchical) == len(flat) == len(target_labels)  # 计算并保存当前步骤的中间状态。
    ):  # 执行当前语句以推进本节示例。
        raise ValueError("训练数据快照长度错误")  # 遇到非法合同立即显式失败。
    split = dataset.get("split", {})  # 计算并保存当前步骤的中间状态。
    if set(split) != {"train", "validation", "test"}:  # 按当前条件选择后续控制路径。
        raise ValueError("训练 split 字段错误")  # 遇到非法合同立即显式失败。
    indices = split["train"] + split["validation"] + split["test"]  # 计算并保存当前步骤的中间状态。
    if len(indices) != len(set(indices)) or sorted(indices) != list(range(len(hierarchical))):  # 按当前条件选择后续控制路径。
        raise ValueError("训练 split 必须互斥且覆盖全部快照")  # 遇到非法合同立即显式失败。
    hierarchy_tensor = torch.tensor(hierarchical, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    flat_tensor = torch.tensor(flat, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    if hierarchy_tensor.ndim != 3 or flat_tensor.ndim != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("训练输入维度错误")  # 遇到非法合同立即显式失败。
    if hierarchy_tensor.numel() and (  # 按当前条件选择后续控制路径。
        int(hierarchy_tensor.min()) < 0 or int(hierarchy_tensor.max()) >= config["vocab_size"]  # 计算并保存当前步骤的中间状态。
    ):  # 执行当前语句以推进本节示例。
        raise ValueError("训练层级 token 越界")  # 遇到非法合同立即显式失败。
    reconstructed_flat = flatten_documents(hierarchy_tensor)  # 计算并保存当前步骤的中间状态。
    if not torch.equal(reconstructed_flat, flat_tensor):  # 按当前条件选择后续控制路径。
        raise ValueError("flat 训练快照不符合绑定的 flatten 规则")  # 遇到非法合同立即显式失败。
    if any(not isinstance(label, int) or not 0 <= label < config["num_classes"] for label in target_labels):  # 按当前条件选择后续控制路径。
        raise ValueError("训练 label 越界")  # 遇到非法合同立即显式失败。
    expected_recipe = {  # 计算并保存当前步骤的中间状态。
        "textcnn": {"optimizer": "Adam", "steps": 70, "lr": 0.025,  # 执行当前语句以推进本节示例。
                    "clip_grad_norm": 2.0, "seed": SEED + 1,  # 执行当前语句以推进本节示例。
                    "objective": "cross_entropy", "purpose": "controlled-overfit"},  # 执行当前语句以推进本节示例。
        "han": {"optimizer": "Adam", "steps": 90, "lr": 0.025,  # 执行当前语句以推进本节示例。
                "clip_grad_norm": 2.0, "seed": SEED + 2,  # 执行当前语句以推进本节示例。
                "objective": "cross_entropy", "purpose": "controlled-overfit"},  # 执行当前语句以推进本节示例。
    }[architecture]  # 执行当前语句以推进本节示例。
    if snapshot.get("recipe") != expected_recipe:  # 按当前条件选择后续控制路径。
        raise ValueError("训练 recipe 快照错误")  # 遇到非法合同立即显式失败。


def package_classifier(model, architecture, vocab_spec, labels, subject,  # 定义本节可复用的核心函数。
                       artifact_id, version, training_snapshot):  # 执行当前语句以推进本节示例。
    if architecture not in {"textcnn", "han"}:  # 按当前条件选择后续控制路径。
        raise ValueError("架构不在白名单")  # 遇到非法合同立即显式失败。
    buffer = io.BytesIO()  # 计算并保存当前步骤的中间状态。
    torch.save(model.state_dict(), buffer)  # 执行当前语句以推进本节示例。
    state_bytes = buffer.getvalue()  # 计算并保存当前步骤的中间状态。
    manifest = {  # 计算并保存当前步骤的中间状态。
        "schema": "document-classifier-v2", "artifact_id": artifact_id,  # 执行当前语句以推进本节示例。
        "version": version, "subject": subject, "architecture": architecture,  # 执行当前语句以推进本节示例。
        "config": copy.deepcopy(model.config), "vocab": copy.deepcopy(vocab_spec),  # 执行当前语句以推进本节示例。
        "label_map": list(labels), "preprocess": expected_preprocess(architecture, model.config),  # 执行当前语句以推进本节示例。
        "training_snapshot": copy.deepcopy(training_snapshot),  # 执行当前语句以推进本节示例。
        "state_bytes_sha256": hashlib.sha256(state_bytes).hexdigest(),  # 执行当前语句以推进本节示例。
        "state_tensor_sha256": classifier_tensor_state_hash(model.state_dict()),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    validate_classifier_manifest(manifest)  # 执行当前语句以推进本节示例。
    return {"manifest": manifest, "manifest_sha256": json_hash(manifest),  # 返回当前分支计算出的结果。
            "state_bytes": state_bytes}  # 执行当前语句以推进本节示例。


vocab_spec = {"kind": "synthetic-id-list-v1", "tokens": CLASSIFIER_TOKENS, "pad_id": PAD}  # 计算并保存当前步骤的中间状态。
label_names = ["技术", "财经", "体育"]  # 计算并保存当前步骤的中间状态。
base_snapshot33 = {  # 计算并保存当前步骤的中间状态。
    "dataset": {"name": "toy-document-classes-v1",  # 执行当前语句以推进本节示例。
                "hierarchical_input_ids": documents.tolist(),  # 执行当前语句以推进本节示例。
                "flat_input_ids": flat_documents.tolist(), "labels": document_labels.tolist(),  # 执行当前语句以推进本节示例。
                "split": {"train": list(range(len(documents))), "validation": [], "test": []}},  # 执行当前语句以推进本节示例。
    "vocab_sha256": json_hash(vocab_spec), "label_map_sha256": json_hash(label_names),  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

def training_snapshot_for(architecture, config):  # 定义本节可复用的核心函数。
    snapshot = copy.deepcopy(base_snapshot33)  # 计算并保存当前步骤的中间状态。
    preprocess = expected_preprocess(architecture, config)  # 计算并保存当前步骤的中间状态。
    snapshot["preprocess_sha256"] = json_hash(preprocess)  # 计算并保存当前步骤的中间状态。
    snapshot["recipe"] = {  # 计算并保存当前步骤的中间状态。
        "textcnn": {"optimizer": "Adam", "steps": 70, "lr": 0.025,  # 执行当前语句以推进本节示例。
                    "clip_grad_norm": 2.0, "seed": SEED + 1,  # 执行当前语句以推进本节示例。
                    "objective": "cross_entropy", "purpose": "controlled-overfit"},  # 执行当前语句以推进本节示例。
        "han": {"optimizer": "Adam", "steps": 90, "lr": 0.025,  # 执行当前语句以推进本节示例。
                "clip_grad_norm": 2.0, "seed": SEED + 2,  # 执行当前语句以推进本节示例。
                "objective": "cross_entropy", "purpose": "controlled-overfit"},  # 执行当前语句以推进本节示例。
    }[architecture]  # 执行当前语句以推进本节示例。
    return snapshot  # 返回当前分支计算出的结果。


han_snapshot = training_snapshot_for("han", han.config)  # 计算并保存当前步骤的中间状态。
textcnn_snapshot = training_snapshot_for("textcnn", textcnn.config)  # 计算并保存当前步骤的中间状态。
han_package = package_classifier(  # 计算并保存当前步骤的中间状态。
    han, "han", vocab_spec, label_names, "doc-team/demo",  # 执行当前语句以推进本节示例。
    "han-demo", "1.0.0", han_snapshot,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
textcnn_package = package_classifier(  # 计算并保存当前步骤的中间状态。
    textcnn, "textcnn", vocab_spec, label_names, "doc-team/demo",  # 执行当前语句以推进本节示例。
    "textcnn-demo", "1.0.0", textcnn_snapshot,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
PUBLISHER_REGISTRY33 = MappingProxyType({  # 计算并保存当前步骤的中间状态。
    ("han-demo", "1.0.0"): han_package["manifest_sha256"],  # 执行当前语句以推进本节示例。
    ("textcnn-demo", "1.0.0"): textcnn_package["manifest_sha256"],  # 执行当前语句以推进本节示例。
})  # 执行当前语句以推进本节示例。


def trusted_classifier_load(package, expected_subject):  # 定义本节可复用的核心函数。
    if not isinstance(package, dict) or set(package) != {"manifest", "manifest_sha256", "state_bytes"}:  # 按当前条件选择后续控制路径。
        raise ValueError("classifier package 字段错误")  # 遇到非法合同立即显式失败。
    manifest = package["manifest"]  # 计算并保存当前步骤的中间状态。
    if not isinstance(manifest, dict):  # 按当前条件选择后续控制路径。
        raise ValueError("manifest 必须是字典")  # 遇到非法合同立即显式失败。
    key = (manifest.get("artifact_id"), manifest.get("version"))  # 计算并保存当前步骤的中间状态。
    expected_digest = PUBLISHER_REGISTRY33.get(key)  # 计算并保存当前步骤的中间状态。
    if expected_digest is None:  # 按当前条件选择后续控制路径。
        raise PermissionError("artifact id/version 未注册")  # 遇到非法合同立即显式失败。
    computed_digest = json_hash(manifest)  # 计算并保存当前步骤的中间状态。
    if package["manifest_sha256"] != computed_digest:  # 按当前条件选择后续控制路径。
        raise ValueError("package 内 manifest hash 不一致")  # 遇到非法合同立即显式失败。
    if computed_digest != expected_digest:  # 按当前条件选择后续控制路径。
        raise PermissionError("package 内容不匹配发布者 registry")  # 遇到非法合同立即显式失败。
    if manifest.get("subject") != expected_subject:  # 按当前条件选择后续控制路径。
        raise PermissionError("业务主体不匹配")  # 遇到非法合同立即显式失败。
    if hashlib.sha256(package["state_bytes"]).hexdigest() != manifest["state_bytes_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("原始 state bytes 指纹不匹配")  # 遇到非法合同立即显式失败。
    validate_classifier_manifest(manifest)  # 执行当前语句以推进本节示例。
    state = torch.load(io.BytesIO(package["state_bytes"]), map_location="cpu", weights_only=True)  # 计算并保存当前步骤的中间状态。
    if classifier_tensor_state_hash(state) != manifest["state_tensor_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("tensor key/dtype/shape/bytes 指纹不匹配")  # 遇到非法合同立即显式失败。
    constructors = {"textcnn": TextCNN, "han": HierarchicalAttentionNetwork}  # 计算并保存当前步骤的中间状态。
    model = constructors[manifest["architecture"]](**manifest["config"])  # 计算并保存当前步骤的中间状态。
    model.load_state_dict(state, strict=True)  # 计算并保存当前步骤的中间状态。
    return model.eval(), manifest["label_map"]  # 返回当前分支计算出的结果。


restored_han, restored_labels = trusted_classifier_load(han_package, "doc-team/demo")  # 计算并保存当前步骤的中间状态。
restored_textcnn, _ = trusted_classifier_load(textcnn_package, "doc-team/demo")  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    assert torch.equal(han(documents, document_word_mask)["logits"],  # 用受控断言验证关键不变量。
                       restored_han(documents, document_word_mask)["logits"])  # 执行当前语句以推进本节示例。
    assert torch.equal(textcnn(flat_documents, flat_mask)[0],  # 用受控断言验证关键不变量。
                       restored_textcnn(flat_documents, flat_mask)[0])  # 执行当前语句以推进本节示例。
assert restored_labels == label_names  # 用受控断言验证关键不变量。

# 模型、完整 token 顺序、label map 即使同步重算内部 hash，也不能越过 registry。
forged_han = HierarchicalAttentionNetwork(**han.config)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    for parameter in forged_han.parameters():  # 遍历输入元素以累积或检查结果。
        parameter.zero_()  # 执行当前语句以推进本节示例。
fully_resigned = package_classifier(  # 计算并保存当前步骤的中间状态。
    forged_han, "han", vocab_spec, label_names, "doc-team/demo",  # 执行当前语句以推进本节示例。
    "han-demo", "1.0.0", han_snapshot,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
resigned_vocab = copy.deepcopy(han_package)  # 计算并保存当前步骤的中间状态。
resigned_vocab["manifest"]["vocab"]["tokens"][4:6] = list(reversed(  # 计算并保存当前步骤的中间状态。
    resigned_vocab["manifest"]["vocab"]["tokens"][4:6]  # 执行当前语句以推进本节示例。
))  # 执行当前语句以推进本节示例。
resigned_vocab["manifest"]["training_snapshot"]["vocab_sha256"] = json_hash(  # 计算并保存当前步骤的中间状态。
    resigned_vocab["manifest"]["vocab"]  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
resigned_vocab["manifest_sha256"] = json_hash(resigned_vocab["manifest"])  # 计算并保存当前步骤的中间状态。
resigned_labels = copy.deepcopy(han_package)  # 计算并保存当前步骤的中间状态。
resigned_labels["manifest"]["label_map"] = ["体育", "财经", "技术"]  # 计算并保存当前步骤的中间状态。
resigned_labels["manifest"]["training_snapshot"]["label_map_sha256"] = json_hash(  # 计算并保存当前步骤的中间状态。
    resigned_labels["manifest"]["label_map"]  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
resigned_labels["manifest_sha256"] = json_hash(resigned_labels["manifest"])  # 计算并保存当前步骤的中间状态。
for candidate in (fully_resigned, resigned_vocab, resigned_labels):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        trusted_classifier_load(candidate, "doc-team/demo")  # 执行当前语句以推进本节示例。
        raise AssertionError("整体重签模型/vocab/label 必须被 registry 拒绝")  # 遇到非法合同立即显式失败。
    except PermissionError as exc:  # 捕获预期异常并验证失败分支。
        assert "registry" in str(exc)  # 用受控断言验证关键不变量。

bad_vocab = copy.deepcopy(vocab_spec)  # 计算并保存当前步骤的中间状态。
bad_vocab["tokens"] = bad_vocab["tokens"][:-1]  # 计算并保存当前步骤的中间状态。
for invalid_vocab, invalid_labels, phrase in [  # 遍历输入元素以累积或检查结果。
    (bad_vocab, label_names, "vocab_size"),  # 执行当前语句以推进本节示例。
    (vocab_spec, ["技术", "财经"], "label"),  # 执行当前语句以推进本节示例。
    (vocab_spec, ["技术", "技术", "体育"], "label"),  # 执行当前语句以推进本节示例。
]:  # 执行当前语句以推进本节示例。
    try:  # 尝试执行可能失败的受控操作。
        package_classifier(  # 执行当前语句以推进本节示例。
            han, "han", invalid_vocab, invalid_labels, "doc-team/demo",  # 执行当前语句以推进本节示例。
            "invalid", "1.0.0", han_snapshot,  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        raise AssertionError("语义不一致制品必须在发布前拒绝")  # 遇到非法合同立即显式失败。
    except ValueError as exc:  # 捕获预期异常并验证失败分支。
        assert phrase in str(exc), str(exc)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    trusted_classifier_load(han_package, "other-team")  # 执行当前语句以推进本节示例。
    raise AssertionError("跨主体加载不应通过")  # 遇到非法合同立即显式失败。
except PermissionError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。
print({"artifact_registry": dict(PUBLISHER_REGISTRY33),  # 执行当前语句以推进本节示例。
       "resigned_model_vocab_label_rejected": True,  # 执行当前语句以推进本节示例。
       "vocab_tokens": len(vocab_spec["tokens"]), "labels": label_names,  # 执行当前语句以推进本节示例。
       "snapshot_documents": len(base_snapshot33["dataset"]["hierarchical_input_ids"])})  # 执行当前语句以推进本节示例。

## 10. 常见失败模式与生产化清单

- TextCNN 对所有窗口直接 max，会把纯 padding 窗口的 bias 选中；必须先做窗口级 mask。
- HAN 将 padding 句送入普通 softmax 会得到均匀分布或 NaN；空行要返回全零权重，全空文档则拒绝。
- 反向 GRU 若先 `flip` 已 padding 序列却不同时处理长度，会让 padding 改变初始状态。
- 词表、分词、最大句数/词数、截断方向和标签顺序都属于模型输入合同，应和 checkpoint 原子发布。
- 长文档截断可能系统性漏掉结论段；上线前按长度分桶评估，并记录被截断比例。
- 监控预测分布、未知词率、长度分布、逐类延迟和漂移；回滚要同时回滚模型与预处理。
- 不把 attention 热力图当成合规解释；重要决策需独立解释与人工复核机制。

## 11. 原始论文与官方资料

- Kim, [Convolutional Neural Networks for Sentence Classification](https://arxiv.org/abs/1408.5882)：多窗口卷积与 max-over-time。
- Yang et al., [Hierarchical Attention Networks for Document Classification](https://aclanthology.org/N16-1174/)：词级/句级层次编码与注意力。
- Cho et al., [Learning Phrase Representations using RNN Encoder–Decoder](https://arxiv.org/abs/1406.1078)：GRU 门控思想。
- PyTorch 官方文档：[Conv1d](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html)、[Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)。

本笔记复现的是结构与正确性合同，并未复现论文数据、词向量、调参预算或报告指标。